# Train a ~200M Fable Transformer (from scratch) — Colab

Keyword-guided fable generation: the model is conditioned on two seed elements
only — **main character** + **moral lesson** — via control-prefix tokens. No base
model, no LoRA, no RAG. Trains a GPT2-style LM from scratch on TF1-EN-3M.

## Run from terminal (google-colab-cli + uv)
```bash
uv tool install google-colab-cli
colab new -s trainer --gpu T4
colab upload -s trainer scripts/prepare_tf1.py /content/scripts/prepare_tf1.py
colab upload -s trainer scripts/fable_tokenizer.py /content/scripts/fable_tokenizer.py
colab upload -s trainer scripts/metrics.py /content/scripts/metrics.py
colab exec -s trainer -f notebooks/train_fable200m_colab.ipynb
colab download -s trainer /content/drive/MyDrive/fable200m ./models/
colab stop -s trainer
```


In [ ]:
# Install deps (uv). If `uv` is unavailable, replace with: !pip install ...
!uv pip install transformers datasets tokenizers accelerate

In [ ]:
import sys, os, json
sys.path.insert(0, '/content')
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
assert os.path.ismount('/content/drive'), 'Drive must be mounted to save the checkpoint'
if os.path.exists('/content/hf_token.txt'):
    os.environ['HF_TOKEN'] = open('/content/hf_token.txt').read().strip()
    print('HF_TOKEN loaded')
else:
    print('No HF_TOKEN (dataset is public; continuing)')
from scripts.prepare_tf1 import iter_tf1, prepare_bpe


In [ ]:
# ---- HYPERPARAMETERS ----
N_FABLES   = 200_000      # full training subset on TF1-EN-3M
VOCAB_SIZE = 8192
BLOCK_SIZE = 1024
# ~200M-param GPT2: n_embd=1024, n_layer=16, n_head=16
MODEL_CFG = dict(
    vocab_size=VOCAB_SIZE, n_positions=BLOCK_SIZE,
    n_embd=1024, n_layer=16, n_head=16,
    resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0,
)
TRAIN_CFG = dict(
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=1,  # overridden by max_steps below
    max_steps=30_000,    # full run (~200M params on 200k fables)
    learning_rate=3e-4,
    warmup_steps=500,
    weight_decay=0.1,
    lr_scheduler_type='cosine',
    bf16=True,
    logging_steps=100,
    save_strategy='steps',
    save_steps=1000,
    seed=42,
)
OUT_DIR = '/content/drive/MyDrive/fable200m'

In [ ]:
# Prepare data: stream TF1, parse (character, moral), train BPE, write fables.jsonl + tokenizer.json
os.makedirs('/content/fable200m', exist_ok=True)
records = list(iter_tf1(N_FABLES, seed=42))
meta = prepare_bpe(records, '/content/fable200m', vocab_size=VOCAB_SIZE)
print(meta)

In [ ]:
from tokenizers import Tokenizer
from transformers import (
    PreTrainedTokenizerFast, GPT2Config, GPT2LMHeadModel,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
)
from datasets import Dataset

tok = Tokenizer.from_file('/content/fable200m/tokenizer.json')
hf_tok = PreTrainedTokenizerFast(tokenizer_object=tok)
hf_tok.pad_token = '</story>'
hf_tok.eos_token = '</story>'

texts = [json.loads(l) for l in open('/content/fable200m/fables.jsonl') if l.strip()]
ds = Dataset.from_dict({'text': texts})
def encode(ex):
    ids = tok.encode(ex['text']).ids
    return {'input_ids': ids, 'attention_mask': [1] * len(ids)}
ds = ds.map(encode, remove_columns=['text']).train_test_split(test_size=0.05)
print(ds)

In [ ]:
cfg = GPT2Config(**MODEL_CFG,
    bos_token_id=tok.get_vocab()['<story>'],
    eos_token_id=tok.get_vocab()['</story>'])
model = GPT2LMHeadModel(cfg)
print('params:', sum(p.numel() for p in model.parameters()))
collator = DataCollatorForLanguageModeling(tokenizer=hf_tok, mlm=False)
args = TrainingArguments(output_dir=OUT_DIR, **TRAIN_CFG)
trainer = Trainer(model=model, args=args,
                  train_dataset=ds['train'], eval_dataset=ds['test'],
                  data_collator=collator)
trainer.train()
trainer.save_model(OUT_DIR)
hf_tok.save_pretrained(OUT_DIR)
print('Saved checkpoint ->', OUT_DIR)

## Export for local app (optional)
Convert the saved checkpoint to GGUF (q8) with `llama.cpp`/`convert_hf_to_gguf.py`
then `ollama create fable-200m -f Modelfile` so the local app's Compare / Results
tabs work unchanged. See `docs/runbooks/colab-train.md`.